# Proposed validation — review before it runs

**What this measures:** `deny_rate_archived_writes` directly exercises `evaluateStateAwareGate` (the actual PR-added predicate) across the 5 resolvable target kinds with `archived=true`, measuring the core "blocks writes to archived resources" claim as a deterministic fraction that must hit 1.0; on baseline the module is absent so the same harness falls back to pre-change semantics (no gate → 0 denials), giving a clean, real Δ.

**Target metric:** `deny_rate_archived_writes`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

The executable copy lives at `eval/eval_state_gate.py`, which is what `.remyx/validation.yaml` points at; keep the two in step, or point `suite:` here if you would rather maintain the notebook.

In [ ]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

## Execution context

The cells below are the script at `eval/eval_state_gate.py`, unchanged. This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [ ]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "eval/eval_state_gate.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

In [ ]:
#!/usr/bin/env python3
"""
eval/eval_state_gate.py

Validates the optional state-aware write gate (state-aware-gate.ts) that runs
after the existing call-only write gate in the Notion MCP proxy:
  - write & target archived  -> must be DENIED
  - write & target live      -> must be ALLOWED
  - read  & target (any)     -> must be ALLOWED, with NO extra round trip

This measures the REAL TypeScript implementation (resolveStateRead,
runStateAwareGate) by driving it from a small generated TS harness executed
with a pinned `tsx` runtime, exactly the way proxy.ts calls it (operation,
params, policy, { resolveReadOperation, readState }). No repo logic is
reimplemented in Python; Python only orchestrates the subprocess and reports
the JSON the TS harness computes.

The "no extra round trip on reads" guardrail is reported as a RATE
(read_no_extra_call_rate = 1 - extra_calls / total_read_cells) rather than a
raw count, so the guardrail threshold is a real, non-vacuous bound (1.0 =
"every read cell incurred zero extra calls") instead of a count-based
threshold of 0 that a min-direction check could never actually fail on.

On the baseline (state-aware-gate.ts absent) the harness's own dynamic
import fails with a module-resolution error, which is the ONLY error this
script tolerates: it then emits pre-change fallback metrics (no gate exists,
so archived writes are never denied, while live writes/reads/no-round-trip
are already correct by construction). Any other failure crashes.
"""
import argparse
import json
import os
import subprocess
import sys
import textwrap

In [ ]:
REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DRIVER_PATH = os.path.join(REPO_ROOT, "eval", "_state_gate_driver.generated.ts")

DRIVER_SRC = textwrap.dedent(
    """
    async function main() {
      const smoke = process.env.REMYX_SMOKE === '1'
      const allKinds = ['page_id', 'block_id', 'database_id', 'data_source_id', 'comment_id']
      const targetKinds = smoke ? allKinds.slice(0, 1) : allKinds
      const policy = { enabled: true, denyWritesToArchived: true }

      let mod: any = null
      let moduleAvailable = true
      try {
        mod = await import('../src/openapi-mcp-server/mcp/state-aware-gate')
      } catch (e: any) {
        const code = e && e.code
        const msg = String((e && e.message) || '')
        if (code === 'ERR_MODULE_NOT_FOUND' || code === 'MODULE_NOT_FOUND' || /Cannot find module/.test(msg)) {
          moduleAvailable = false
        } else {
          throw e
        }
      }

      if (!moduleAvailable) {
        // Pre-change fallback: no state gate exists at all. Archived writes
        // are never denied (0/5). Live writes and reads were already
        // allowed and never round-tripped, so those metrics are unaffected.
        console.log(JSON.stringify({
          deny_rate_archived_writes: 0.0,
          allow_rate_live_writes: 1.0,
          read_invariance_rate: 1.0,
          read_no_extra_call_rate: 1.0,
        }))
        return
      }

      const { resolveStateRead, runStateAwareGate } = mod

      let writeArchivedDenied = 0
      let writeLiveAllowed = 0
      let readAllowed = 0
      let extraApiCallsOnReads = 0

      for (const kind of targetKinds) {
        const params: Record<string, string> = { [kind]: `${kind}-target-1` }
        const resolved = resolveStateRead(params)
        if (!resolved) {
          throw new Error(`resolveStateRead did not resolve declared target kind '${kind}' (fixture assumption from the mechanism spec is invalid)`) 
        }
        const readOp = { operationId: resolved.operationId, responses: { '200': { description: 'Success' } }, method: 'get', path: `/${kind}/{id}` }
        const writeOp = { operationId: `update-${kind}`, responses: { '200': { description: 'Success' } }, method: 'patch', path: `/${kind}/{id}` }

        for (const archived of [true, false]) {
          const writeDeps = {
            resolveReadOperation: (id: string) => (id === readOp.operationId ? readOp : null),
            readState: async (_op: unknown, _p: unknown) => ({ archived }),
          }
          const writeDecision = await runStateAwareGate(writeOp, params, policy, writeDeps)
          if (archived && writeDecision.allowed === false) writeArchivedDenied += 1
          if (!archived && writeDecision.allowed === true) writeLiveAllowed += 1

          let readRoundTrips = 0
          const readDeps = {
            resolveReadOperation: (id: string) => (id === readOp.operationId ? readOp : null),
            readState: async (_op: unknown, _p: unknown) => {
              readRoundTrips += 1
              return { archived }
            },
          }
          const readDecision = await runStateAwareGate(readOp, params, policy, readDeps)
          if (readDecision.allowed === true) readAllowed += 1
          extraApiCallsOnReads += readRoundTrips
        }
      }

      const totalWriteCells = targetKinds.length
      const totalReadCells = targetKinds.length * 2
      // Real bound: on a correctly implemented read path, every one of the
      // totalReadCells read decisions should incur zero readState round
      // trips, so this rate should be exactly 1.0. Any extra call pulls it
      // below 1.0, which the fallback arm above also achieves (1.0),
      // making this a genuine, satisfiable-by-baseline guardrail.
      const readNoExtraCallRate = (totalReadCells - extraApiCallsOnReads) / totalReadCells
      console.log(JSON.stringify({
        deny_rate_archived_writes: writeArchivedDenied / totalWriteCells,
        allow_rate_live_writes: writeLiveAllowed / totalWriteCells,
        read_invariance_rate: readAllowed / totalReadCells,
        read_no_extra_call_rate: readNoExtraCallRate,
      }))
    }

    main().catch((e) => {
      console.error(e)
      process.exit(1)
    })
    """
).strip() + "\n"

In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--variant", default=None)
    parser.add_argument("--ref", default=None)
    parser.add_argument("--seed", default=None)
    parser.parse_known_args()

    os.makedirs(os.path.dirname(DRIVER_PATH), exist_ok=True)
    with open(DRIVER_PATH, "w") as f:
        f.write(DRIVER_SRC)

    # Pinned tsx runtime (held_constant) so both arms execute the TS module
    # with the identical, deterministic toolchain.
    proc = subprocess.run(
        ["npx", "-y", "tsx@4.19.0", DRIVER_PATH],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        env=os.environ.copy(),
    )
    if proc.returncode != 0:
        raise RuntimeError(
            f"state-aware-gate driver failed (exit {proc.returncode}):\n"
            f"--- stdout ---\n{proc.stdout}\n--- stderr ---\n{proc.stderr}"
        )

    json_lines = [line for line in proc.stdout.splitlines() if line.strip().startswith("{")]
    if not json_lines:
        raise RuntimeError(f"no JSON line produced by driver; stdout was:\n{proc.stdout}\nstderr:\n{proc.stderr}")

    metrics = json.loads(json_lines[-1])
    required = {
        "deny_rate_archived_writes",
        "allow_rate_live_writes",
        "read_invariance_rate",
        "read_no_extra_call_rate",
    }
    missing = required - metrics.keys()
    if missing:
        raise RuntimeError(f"driver output missing required keys: {missing} (got {metrics})")

    print(json.dumps(metrics))

if __name__ == "__main__":
    main()

## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
loop: {max_iterations: 8, fix_code: true}
benchmarks:
  - name: state-aware-write-gate
    suite: "eval/eval_state_gate.py"
    scorer: deny_rate_archived_writes
    baseline: main
    metrics:
      - name: deny_rate_archived_writes
        role: target
        direction: max
        threshold: 1.0
      - name: allow_rate_live_writes
        role: guardrail
        direction: max
        threshold: 1.0
      - name: read_invariance_rate
        role: guardrail
        direction: max
        threshold: 1.0
      - name: read_no_extra_call_rate
        role: guardrail
        direction: max
        threshold: 1.0
    policy: {guardrail_veto: true}
held_constant:
  - "same 5 resolvable target-kind params across arms: page_id, block_id, database_id, data_source_id, comment_id"
  - "same synthetic OpenAPIV3 write (patch) and read (get) operation stubs, matching the PR test file's patchOp/getOp helper shapes"
  - "same explicitly-constructed policy object {enabled: true, denyWritesToArchived: true} passed directly to evaluateStateAwareGate, bypassing env-var defaults so the feature is exercised ON regardless of the arm"
  - "same pinned tsx version (4.19.0) used to execute the TypeScript module on both arms for determinism"
avoid:
  - "no reliance on a live Notion API: all read/write targets are synthetic OpenAPIV3 operation objects, never real HTTP calls"
  - "no wall-clock timing anywhere in the metric computation"
  - "do not let npx's own download/log chatter leak into the final JSON line; only the last stdout line is treated as the result"
compute: {tier: cpu}
provenance:
  deny_rate_archived_writes: "user_guidance"
  allow_rate_live_writes: "claim_analysis:recommended.metrics"
  read_invariance_rate: "user_guidance"
  read_no_extra_call_rate: "claim_analysis:mismatches (reads must not round-trip); reported as a rate (1.0 = every read cell incurred zero extra readState calls) so the guardrail threshold of 1.0 is a real, satisfiable-by-baseline bound instead of a count-based threshold of 0 that a min-direction check could never fail on"
  held_constant: "repo_anchor:src/openapi-mcp-server/mcp/__tests__/state-aware-gate.test.ts"
  suite: "synthesized"
  avoid: "inferred"
```